# H2 evaluation: three-band versus six-band multispectral inputs

Evaluates matched three-band and six-band models under the leave-one-site-out design. Both arms use the multispectral pathway, with validation-selected thresholds applied unchanged to each held-out site.

## Setup

Installs the fixed Detectron2 and Detectree2 revisions used in the analysis. Restart the Colab runtime after installation.

In [ ]:
!pip -q install \
    "git+https://github.com/facebookresearch/detectron2.git@a2f4a8771ab77e8411c26b27f24f9489a28a2453"

!pip -q install \
    "git+https://github.com/PatBall1/detectree2.git@d9fb07f0dfb493f34def563c1ff896fecd59210d"

!pip -q install rasterio geopandas pyyaml

## Paths and protocol

Defines the input, checkpoint, prediction-cache and output paths together with the shared evaluation parameters.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import gc
import json
import shutil

import cv2
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
import yaml

from rasterio.mask import mask as rio_mask
from rasterio.transform import xy as raster_xy
from shapely.geometry import Polygon
from shapely.ops import unary_union

from detectron2.data import (
    DatasetCatalog,
    MetadataCatalog,
)
from detectron2.engine import DefaultPredictor

from detectree2.models.train import (
    get_tree_dicts,
    setup_cfg,
)
from detectree2.preprocessing.tiling import tile_data


DATA_ROOT = Path(
    "/content/drive/MyDrive/Congo basin/congo"
)
H2_RUNS = Path(
    "/content/drive/MyDrive/Congo basin/H2_runs"
)
H1_OUTPUTS = DATA_ROOT / "h1_evaluation"

OUTPUTS = H2_RUNS / "evaluation_ms"
PREDICTION_CACHE = (
    OUTPUTS
    / "candidate_predictions"
)

WORK = Path("/content/h2_evaluation_work")
STACKS = WORK / "stacks"
CROPS = WORK / "crops"
TILES = WORK / "tiles"
KEEP = WORK / "keep"

REBUILD_INPUTS = False
RECOMPUTE_PREDICTIONS = False

if REBUILD_INPUTS and WORK.exists():
    shutil.rmtree(WORK)

for directory in (
    STACKS,
    CROPS,
    TILES,
    KEEP,
    OUTPUTS,
    PREDICTION_CACHE,
):
    directory.mkdir(parents=True, exist_ok=True)


SITES = ["lokoue", "dzanga", "mbeli"]
STRATA = ["tall", "mid", "small"]

SITE_LABELS = {
    "lokoue": "Lokoué",
    "dzanga": "Dzanga",
    "mbeli": "Mbeli",
}

STRATUM_LABELS = {
    "tall": "High",
    "mid": "Intermediate",
    "small": "Low",
}

ARMS = {
    "three_band": {
        "label": "Three-band MS",
        "paper_label": "RGB H2",
        "num_bands": 3,
        "run_prefix": "rgb",
    },
    "six_band": {
        "label": "Six-band MS",
        "paper_label": "Six-band",
        "num_bands": 6,
        "run_prefix": "ms",
    },
}

TILE_WIDTH = 50
BUFFER = 25

CANDIDATE_THRESHOLD = 0.05
MATCHING_IOU = 0.50

CONFIDENCE_THRESHOLDS = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
]

NMS_THRESHOLDS = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
]

BASE_MODEL = (
    "COCO-InstanceSegmentation/"
    "mask_rcnn_R_101_FPN_3x.yaml"
)

MIN_CHECKPOINT_MB = 400

paths = {
    site: {
        "rgb": (
            DATA_ROOT
            / site
            / f"{site}_RGB.TIF"
        ),
        "ned": (
            DATA_ROOT
            / site
            / f"{site}_NED.TIF"
        ),
        "stack": (
            STACKS
            / f"{site}_six_band_uint16.tif"
        ),
        "mask": (
            DATA_ROOT
            / site
            / f"mask_{site}.gpkg"
        ),
        "aoi": {
            stratum: (
                DATA_ROOT
                / site
                / f"{site}_aoi_{stratum}.gpkg"
            )
            for stratum in STRATA
        },
        "crowns": {
            stratum: (
                DATA_ROOT
                / site
                / f"crowns_{site}_{stratum}.gpkg"
            )
            for stratum in STRATA
        },
    }
    for site in SITES
}

## Checkpoint audit

Checks that every H2 checkpoint and metrics file is present and that checkpoint input channels match the declared arm. Archived configurations are additionally checked for multispectral mode; legacy runs without config.yaml are reported explicitly.

In [ ]:
def run_directory(arm, holdout):
    """Return the saved-model directory for one H2 arm and fold."""
    prefix = ARMS[arm]["run_prefix"]
    return H2_RUNS / f"{prefix}_{holdout}"


def checkpoint_path(arm, holdout):
    """Return the selected checkpoint for one H2 arm and fold."""
    return run_directory(arm, holdout) / "model_best.pth"


def checkpoint_input_channels(path):
    """Read the number of input channels encoded in a checkpoint."""
    checkpoint = torch.load(
        path,
        map_location="cpu",
    )
    state_dict = checkpoint.get(
        "model",
        checkpoint,
    )

    convolution_keys = [
        key
        for key in state_dict
        if key.endswith("stem.conv1.weight")
    ]

    assert len(convolution_keys) == 1, (
        f"{path}: expected one stem.conv1.weight tensor, "
        f"found {convolution_keys}"
    )

    channels = int(
        state_dict[
            convolution_keys[0]
        ].shape[1]
    )

    del checkpoint, state_dict
    gc.collect()

    return channels


def best_ap50_iteration(metrics_path):
    """Return the highest recorded validation AP50 and its iteration."""
    assert metrics_path.exists(), (
        f"Missing metrics file: {metrics_path}"
    )

    evaluations = []

    with metrics_path.open() as stream:
        for line in stream:
            record = json.loads(line)

            if "segm/AP50" in record:
                evaluations.append(
                    {
                        "iteration": int(
                            record["iteration"]
                        ),
                        "ap50": float(
                            record["segm/AP50"]
                        ),
                    }
                )

    assert evaluations, (
        f"No segm/AP50 records found in {metrics_path}"
    )

    return max(
        evaluations,
        key=lambda record: (
            record["ap50"],
            -record["iteration"],
        ),
    )


audit_rows = []

for arm, settings in ARMS.items():
    for holdout in SITES:
        directory = run_directory(
            arm,
            holdout,
        )
        checkpoint = checkpoint_path(
            arm,
            holdout,
        )
        configuration_path = (
            directory / "config.yaml"
        )
        metrics_path = (
            directory / "metrics.json"
        )

        assert checkpoint.exists(), (
            f"Missing checkpoint: {checkpoint}"
        )
        assert checkpoint.stat().st_size > 0, (
            f"Empty checkpoint: {checkpoint}"
        )
        assert metrics_path.exists(), (
            f"Missing metrics file: {metrics_path}"
        )

        expected_bands = int(
            settings["num_bands"]
        )
        checkpoint_bands = (
            checkpoint_input_channels(
                checkpoint
            )
        )

        assert checkpoint_bands == expected_bands, (
            f"{arm}/{holdout}: checkpoint has "
            f"{checkpoint_bands} input channels; "
            f"expected {expected_bands}"
        )

        if configuration_path.exists():
            with configuration_path.open() as stream:
                configuration = (
                    yaml.safe_load(stream)
                )

            assert isinstance(
                configuration,
                dict,
            ), (
                f"{configuration_path}: "
                "configuration is not a mapping"
            )

            saved_mode = str(
                configuration.get(
                    "IMGMODE",
                    "",
                )
            ).lower()

            saved_bands = int(
                configuration.get(
                    "INPUT",
                    {},
                ).get(
                    "NUM_IN_CHANNELS",
                    -1,
                )
            )

            assert saved_mode == "ms", (
                f"{arm}/{holdout}: saved IMGMODE is "
                f"{saved_mode!r}, expected 'ms'"
            )
            assert int(saved_bands) == expected_bands, (
                f"{arm}/{holdout}: saved NUM_BANDS is "
                f"{saved_bands}, expected {expected_bands}"
            )

            configuration_status = "verified"

        else:
            saved_mode = None
            saved_bands = None
            configuration_status = "not archived"

            print(
                f"NOTE: {arm}/{holdout} has no archived "
                "config.yaml; checkpoint channels and "
                "metrics were verified instead."
            )

        best_record = best_ap50_iteration(
            metrics_path
        )

        audit_rows.append(
            {
                "arm": arm,
                "held_out_site": holdout,
                "declared_imgmode": "ms",
                "declared_bands": expected_bands,
                "saved_imgmode": saved_mode,
                "saved_bands": saved_bands,
                "configuration_status": (
                    configuration_status
                ),
                "checkpoint_bands": (
                    checkpoint_bands
                ),
                "best_validation_ap50": (
                    best_record["ap50"]
                ),
                "best_iteration": (
                    best_record["iteration"]
                ),
                "checkpoint": str(checkpoint),
            }
        )


model_audit_table = pd.DataFrame(
    audit_rows
)

assert len(model_audit_table) == 6, (
    "Expected six H2 model-audit records"
)

display(model_audit_table)

## Input validation

Checks the source rasters and spatial annotations before reconstructing the evaluation inputs.

In [ ]:
required_files = []

for site in SITES:
    required_files.extend([
        paths[site]["rgb"],
        paths[site]["ned"],
        paths[site]["mask"],
        *paths[site]["aoi"].values(),
        *paths[site]["crowns"].values(),
    ])

missing_files = [
    path
    for path in required_files
    if not path.exists()
]

assert not missing_files, (
    "Missing source files:\n"
    + "\n".join(map(str, missing_files))
)

for site in SITES:
    with (
        rasterio.open(paths[site]["rgb"]) as rgb,
        rasterio.open(paths[site]["ned"]) as ned,
    ):
        assert rgb.count >= 3 and ned.count >= 3
        assert rgb.crs == ned.crs
        assert rgb.shape == ned.shape
        assert rgb.transform == ned.transform
        assert rgb.bounds == ned.bounds
        assert rgb.res == ned.res
        assert rgb.dtypes[:3] == ned.dtypes[:3]
        assert set(rgb.dtypes[:3]) == {"uint16"}

        print(
            f"{site}: {rgb.width} × {rgb.height}, "
            f"{rgb.res[0]:.3f} m, {rgb.crs}"
        )

## Six-band inputs

Reconstructs the native 16-bit six-band stacks used during H2 training.

In [ ]:
def build_six_band_stack(site):
    output_path = paths[site]["stack"]

    if output_path.exists() and not REBUILD_INPUTS:
        with rasterio.open(output_path) as raster:
            valid_output = (
                raster.count == 6
                and raster.dtypes == ("uint16",) * 6
                and raster.nodata == 0
            )

        if valid_output:
            return output_path

        output_path.unlink()

    with (
        rasterio.open(paths[site]["rgb"]) as rgb,
        rasterio.open(paths[site]["ned"]) as ned,
    ):
        metadata = rgb.meta.copy()
        metadata.update(
            driver="GTiff",
            count=6,
            dtype="uint16",
            nodata=0,
            tiled=True,
            blockxsize=512,
            blockysize=512,
            compress="deflate",
        )

        with rasterio.open(
            output_path,
            "w",
            **metadata,
        ) as destination:
            for _, window in rgb.block_windows(1):
                values = np.concatenate([
                    rgb.read(
                        [1, 2, 3],
                        window=window,
                    ),
                    ned.read(
                        [1, 2, 3],
                        window=window,
                    ),
                ], axis=0)

                destination.write(
                    values,
                    window=window,
                )

    with rasterio.open(output_path) as raster:
        assert raster.count == 6
        assert raster.dtypes == ("uint16",) * 6
        assert raster.nodata == 0

    return output_path


for site in SITES:
    build_six_band_stack(site)

print("Six-band stacks ready")

## Valid evaluation regions

Reconstructs the same AOI-minus-mask regions used during training and H1 evaluation.

In [ ]:
prepped = {}

for site in SITES:
    with rasterio.open(paths[site]["stack"]) as raster:
        raster_crs = raster.crs

    boundary_mask = gpd.read_file(
        paths[site]["mask"]
    ).to_crs(raster_crs)
    boundary_mask.geometry = (
        boundary_mask.geometry.buffer(0)
    )

    prepped[site] = {}

    for stratum in STRATA:
        aoi = gpd.read_file(
            paths[site]["aoi"][stratum]
        ).to_crs(raster_crs)

        crowns = gpd.read_file(
            paths[site]["crowns"][stratum]
        ).to_crs(raster_crs)

        aoi.geometry = aoi.geometry.buffer(0)
        crowns.geometry = crowns.geometry.buffer(0)

        assert aoi.geometry.is_valid.all()
        assert crowns.geometry.is_valid.all()

        assert (
            crowns.geometry.centroid.within(
                unary_union(aoi.geometry)
            )
        ).all()

        keep = gpd.overlay(
            aoi,
            boundary_mask,
            how="difference",
        )
        keep.geometry = keep.geometry.buffer(0)

        keep_path = (
            KEEP
            / f"keep_{site}_{stratum}.gpkg"
        )
        keep.to_file(
            keep_path,
            driver="GPKG",
        )

        keep_union = unary_union(keep.geometry)

        evaluation_crowns = crowns[
            crowns.geometry.centroid.within(
                keep_union
            )
        ].copy()

        prepped[site][stratum] = {
            "crowns": crowns,
            "evaluation_crowns": evaluation_crowns,
            "keep": keep,
            "keep_path": keep_path,
        }

## Six-band tiling

Recreates the multispectral tiles used during H2 training.

In [ ]:
def tile_six_band_aoi(site, stratum):
    crowns = prepped[site][stratum]["crowns"]
    keep_path = prepped[site][stratum]["keep_path"]

    aoi = gpd.read_file(
        paths[site]["aoi"][stratum]
    ).to_crs(crowns.crs)

    crop_path = (
        CROPS
        / f"{site}_{stratum}_six_band.tif"
    )

    if not crop_path.exists():
        with rasterio.open(
            paths[site]["stack"]
        ) as source:
            image, transform = rio_mask(
                source,
                aoi.geometry.buffer(
                    2,
                    join_style=2,
                ),
                crop=True,
                nodata=0,
            )

            metadata = source.meta.copy()
            metadata.update(
                height=image.shape[1],
                width=image.shape[2],
                transform=transform,
                nodata=0,
            )

        with rasterio.open(
            crop_path,
            "w",
            **metadata,
        ) as destination:
            destination.write(image)

    output_directory = (
        TILES
        / f"{site}_{stratum}_six_band_"
          f"{TILE_WIDTH}_{BUFFER}"
    )

    if not output_directory.exists():
        tile_data(
            img_path=str(crop_path),
            out_dir=str(output_directory),
            buffer=BUFFER,
            tile_width=TILE_WIDTH,
            tile_height=TILE_WIDTH,
            crowns=crowns,
            threshold=0.0,
            nan_threshold=1.0,
            full_coverage=False,
            mode="ms",
            mask_path=str(keep_path),
            use_convex_mask=False,
            enhance_rgb_contrast=False,
            tile_placement="grid",
            multithreaded=True,
            ignore_bands_indices=[],
        )

    assert (
        len(list(output_directory.glob("*.tif")))
        == 25
    )
    assert (
        len(list(output_directory.glob("*.geojson")))
        == 25
    )

    return output_directory


six_band_directories = {
    (site, stratum): tile_six_band_aoi(
        site,
        stratum,
    )
    for site in SITES
    for stratum in STRATA
}

assert sum(
    len(list(directory.glob("*.geojson")))
    for directory in six_band_directories.values()
) == 225

## Three-band multispectral inputs

Recreates the three-band GeoTIFF clones while retaining the multispectral loading route.

In [ ]:
def create_three_band_ms_clone(
    source_directory,
    output_directory,
):
    if output_directory.exists():
        assert (
            len(list(output_directory.glob("*.tif")))
            == 25
        )
        assert (
            len(list(output_directory.glob("*.geojson")))
            == 25
        )

        return output_directory

    output_directory.mkdir(
        parents=True,
        exist_ok=False,
    )

    source_rasters = sorted(
        source_directory.glob("*.tif")
    )

    assert len(source_rasters) == 25

    for source_raster in source_rasters:
        output_raster = (
            output_directory
            / source_raster.name
        )

        with rasterio.open(source_raster) as source:
            data = source.read([1, 2, 3])
            metadata = source.meta.copy()
            metadata.update(count=3)

        with rasterio.open(
            output_raster,
            "w",
            **metadata,
        ) as destination:
            destination.write(data)

        source_annotation = (
            source_directory
            / f"{source_raster.stem}.geojson"
        )
        output_annotation = (
            output_directory
            / source_annotation.name
        )

        assert source_annotation.exists()

        with open(
            source_annotation,
            "r",
        ) as annotation_file:
            annotation = json.load(
                annotation_file
            )

        annotation["imagePath"] = str(
            output_raster.resolve()
        )

        with open(
            output_annotation,
            "w",
        ) as annotation_file:
            json.dump(
                annotation,
                annotation_file,
                indent=2,
            )

    assert (
        len(list(output_directory.glob("*.tif")))
        == 25
    )
    assert (
        len(list(output_directory.glob("*.geojson")))
        == 25
    )

    return output_directory


three_band_directories = {}

for site in SITES:
    for stratum in STRATA:
        output_directory = (
            TILES
            / f"{site}_{stratum}_three_band_ms_"
              f"{TILE_WIDTH}_{BUFFER}"
        )

        three_band_directories[
            (site, stratum)
        ] = create_three_band_ms_clone(
            six_band_directories[
                (site, stratum)
            ],
            output_directory,
        )

assert sum(
    len(list(directory.glob("*.geojson")))
    for directory in three_band_directories.values()
) == 225

## Matched-input verification

Verifies pixel, spatial-metadata and annotation identity across all 225 tile pairs.

In [ ]:
verified_pairs = 0

for site in SITES:
    for stratum in STRATA:
        three_directory = three_band_directories[
            (site, stratum)
        ]
        six_directory = six_band_directories[
            (site, stratum)
        ]

        six_rasters = {
            path.stem: path
            for path in six_directory.glob("*.tif")
        }

        for three_path in sorted(
            three_directory.glob("*.tif")
        ):
            six_path = six_rasters[
                three_path.stem
            ]

            with (
                rasterio.open(three_path) as three,
                rasterio.open(six_path) as six,
            ):
                assert three.count == 3
                assert six.count == 6
                assert three.width == six.width
                assert three.height == six.height
                assert three.transform == six.transform
                assert three.crs == six.crs
                assert three.nodata == six.nodata
                assert three.dtypes == six.dtypes[:3]

                assert np.array_equal(
                    three.read(),
                    six.read([1, 2, 3]),
                )

            three_annotation_path = (
                three_directory
                / f"{three_path.stem}.geojson"
            )
            six_annotation_path = (
                six_directory
                / f"{three_path.stem}.geojson"
            )

            with open(
                three_annotation_path,
                "r",
            ) as annotation_file:
                three_annotation = json.load(
                    annotation_file
                )

            with open(
                six_annotation_path,
                "r",
            ) as annotation_file:
                six_annotation = json.load(
                    annotation_file
                )

            three_labels = {
                key: value
                for key, value in three_annotation.items()
                if key != "imagePath"
            }
            six_labels = {
                key: value
                for key, value in six_annotation.items()
                if key != "imagePath"
            }

            assert three_labels == six_labels

            linked_image = Path(
                three_annotation["imagePath"]
            )

            assert linked_image.resolve() == (
                three_path.resolve()
            )
            assert linked_image.suffix.lower() == ".tif"

            verified_pairs += 1

assert verified_pairs == 225

print(
    "PASS: 225 matched tile pairs verified"
)

## Leave-one-site-out datasets

Registers both H2 arms and verifies the training, validation and held-out partitions.

In [ ]:
arm_directories = {
    "three_band": three_band_directories,
    "six_band": six_band_directories,
}


def register_fold(
    directories,
    holdout,
    tag,
):
    names = {
        split: f"{tag}_{split}"
        for split in ("train", "validation", "test")
    }

    for name in names.values():
        if name in DatasetCatalog.list():
            DatasetCatalog.remove(name)

        if name in MetadataCatalog.list():
            MetadataCatalog.remove(name)

    partitions = {
        "train": [],
        "validation": [],
        "test": [],
    }

    for (site, stratum), directory in directories.items():
        records = get_tree_dicts(str(directory))

        if site == holdout:
            partitions["test"].extend(records)
        elif stratum == "mid":
            partitions["validation"].extend(records)
        else:
            partitions["train"].extend(records)

    counts = {
        split: len(records)
        for split, records in partitions.items()
    }

    assert counts == {
        "train": 100,
        "validation": 50,
        "test": 75,
    }

    for split, records in partitions.items():
        DatasetCatalog.register(
            names[split],
            lambda records=records: records,
        )

        MetadataCatalog.get(
            names[split]
        ).set(thing_classes=["tree"])

    return names


registered_folds = {}

for arm in ARMS:
    for holdout in SITES:
        registered_folds[
            (arm, holdout)
        ] = register_fold(
            arm_directories[arm],
            holdout,
            f"h2_evaluation_{arm}_{holdout}",
        )

print("PASS: six evaluation folds registered")

## Geographic reconstruction and matching

Uses the same polygon reconstruction, NMS and one-to-one matching implementation as H1 evaluation.

In [ ]:
def intersection_over_union(first, second):
    if not first.intersects(second):
        return 0.0

    intersection = first.intersection(second).area
    union = (
        first.area
        + second.area
        - intersection
    )

    return intersection / union if union else 0.0


def non_maximum_suppression(predictions, threshold):
    retained = []

    ordered = sorted(
        predictions,
        key=lambda prediction: prediction["score"],
        reverse=True,
    )

    for prediction in ordered:
        duplicate = any(
            intersection_over_union(
                prediction["geometry"],
                retained_prediction["geometry"],
            ) >= threshold
            for retained_prediction in retained
        )

        if not duplicate:
            retained.append(prediction)

    return retained


def greedy_matches(
    predictions,
    references,
    threshold=MATCHING_IOU,
):
    eligible_pairs = []

    for prediction_index, prediction in enumerate(predictions):
        for reference_index, reference in enumerate(references):
            overlap = intersection_over_union(
                prediction,
                reference,
            )

            if overlap >= threshold:
                eligible_pairs.append((
                    overlap,
                    prediction_index,
                    reference_index,
                ))

    eligible_pairs.sort(reverse=True)

    used_predictions = set()
    used_references = set()
    matches = []

    for overlap, prediction_index, reference_index in eligible_pairs:
        if (
            prediction_index in used_predictions
            or reference_index in used_references
        ):
            continue

        used_predictions.add(prediction_index)
        used_references.add(reference_index)

        matches.append((
            overlap,
            prediction_index,
            reference_index,
        ))

    return matches


def precision_recall_f1(tp, fp, fn):
    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )
    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )
    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    return precision, recall, f1


def mask_to_map_polygons(binary_mask, transform):
    contours, _ = cv2.findContours(
        binary_mask.astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE,
    )

    polygons = []

    for contour in contours:
        if len(contour) < 3:
            continue

        coordinates = contour.reshape(-1, 2)

        x, y = raster_xy(
            transform,
            coordinates[:, 1],
            coordinates[:, 0],
            offset="center",
        )

        polygon = Polygon(
            zip(
                np.atleast_1d(x),
                np.atleast_1d(y),
            )
        ).buffer(0)

        if not polygon.is_empty and polygon.area > 0:
            polygons.append(polygon)

    return polygons

## References and valid regions

Loads each reference crown once and restricts predictions and references to the same valid areas.

In [ ]:
def reference_records_for_aois(aois):
    records = []

    for site, stratum in aois:
        crowns = prepped[site][stratum][
            "evaluation_crowns"
        ]

        for source_index, geometry in zip(
            crowns.index,
            crowns.geometry,
        ):
            records.append({
                "site": site,
                "stratum": stratum,
                "source_index": int(source_index),
                "geometry": geometry,
            })

    return records


def keep_region_for_aois(aois):
    return unary_union([
        geometry.buffer(0)
        for site, stratum in aois
        for geometry in prepped[
            site
        ][stratum]["keep"].geometry
    ])


def evaluate_at(
    predictions,
    reference_records,
    keep_region,
    confidence_threshold,
    nms_threshold,
):
    confidence_filtered = [
        prediction
        for prediction in predictions
        if prediction["score"]
        >= confidence_threshold
    ]

    deduplicated = non_maximum_suppression(
        confidence_filtered,
        nms_threshold,
    )

    retained = [
        prediction
        for prediction in deduplicated
        if keep_region.contains(
            prediction["geometry"].centroid
        )
    ]

    references = [
        record["geometry"]
        for record in reference_records
    ]

    matches = greedy_matches(
        [
            prediction["geometry"]
            for prediction in retained
        ],
        references,
    )

    true_positives = len(matches)
    false_positives = (
        len(retained) - true_positives
    )
    false_negatives = (
        len(references) - true_positives
    )

    precision, recall, f1 = precision_recall_f1(
        true_positives,
        false_positives,
        false_negatives,
    )

    assert true_positives + false_positives == len(
        retained
    )
    assert true_positives + false_negatives == len(
        references
    )

    return {
        "confidence": confidence_threshold,
        "nms": nms_threshold,
        "tp": true_positives,
        "fp": false_positives,
        "fn": false_negatives,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

## Multispectral prediction and caching

Runs both arms from GeoTIFF inputs at a candidate confidence threshold of 0.05 and caches the mapped predictions.

In [ ]:
def build_predictor(arm, holdout):
    settings = ARMS[arm]
    names = registered_folds[
        (arm, holdout)
    ]

    configuration = setup_cfg(
        base_model=BASE_MODEL,
        trains=(names["train"],),
        tests=(names["validation"],),
        update_model=None,
        workers=2,
        ims_per_batch=2,
        max_iter=1,
        eval_period=1,
        resize="rand_fixed",
        imgmode="ms",
        num_bands=settings["num_bands"],
        out_dir=(
            f"/content/h2_predictor_"
            f"{arm}_{holdout}"
        ),
    )

    configuration.MODEL.WEIGHTS = str(
        checkpoint_path(arm, holdout)
    )
    configuration.MODEL.ROI_HEADS.SCORE_THRESH_TEST = (
        CANDIDATE_THRESHOLD
    )

    return DefaultPredictor(configuration)


def prediction_cache_path(
    arm,
    holdout,
    site,
    stratum,
):
    return (
        PREDICTION_CACHE
        / f"{arm}_ms_holdout_{holdout}_"
          f"{site}_{stratum}.gpkg"
    )


def read_prediction_cache(
    arm,
    holdout,
    site,
    stratum,
):
    predictions = gpd.read_file(
        prediction_cache_path(
            arm,
            holdout,
            site,
            stratum,
        )
    )

    return [
        {
            "geometry": row.geometry,
            "score": float(row.score),
            "site": row.site,
            "stratum": row.stratum,
            "tile": row.tile,
        }
        for row in predictions.itertuples()
    ]


def predict_aoi(
    predictor,
    arm,
    holdout,
    site,
    stratum,
):
    cache_path = prediction_cache_path(
        arm,
        holdout,
        site,
        stratum,
    )

    if (
        cache_path.exists()
        and not RECOMPUTE_PREDICTIONS
    ):
        print(
            f"{arm}/{holdout}/{site}/{stratum}: "
            "using cached predictions"
        )

        return read_prediction_cache(
            arm,
            holdout,
            site,
            stratum,
        )

    predictions = []
    output_crs = None

    records = get_tree_dicts(
        str(
            arm_directories[arm][
                (site, stratum)
            ]
        )
    )

    for record in records:
        image_path = Path(record["file_name"])

        assert image_path.suffix.lower() == ".tif"
        assert image_path.exists()

        with rasterio.open(image_path) as raster:
            assert raster.count == ARMS[
                arm
            ]["num_bands"]

            transform = raster.transform
            output_crs = raster.crs

            image = np.transpose(
                raster.read().astype(np.float32),
                (1, 2, 0),
            )

        assert image.shape[2] == ARMS[
            arm
        ]["num_bands"]
        assert np.isfinite(image).all()

        instances = predictor(image)[
            "instances"
        ].to("cpu")

        for mask, score in zip(
            instances.pred_masks.numpy(),
            instances.scores.numpy(),
        ):
            for polygon in mask_to_map_polygons(
                mask.astype(bool),
                transform,
            ):
                predictions.append({
                    "geometry": polygon,
                    "score": float(score),
                    "site": site,
                    "stratum": stratum,
                    "tile": image_path.stem,
                })

    assert predictions, (
        f"{arm}/{holdout}/{site}/{stratum}: "
        "no candidate predictions"
    )

    prediction_table = gpd.GeoDataFrame(
        {
            "score": [
                prediction["score"]
                for prediction in predictions
            ],
            "site": site,
            "stratum": stratum,
            "tile": [
                prediction["tile"]
                for prediction in predictions
            ],
        },
        geometry=[
            prediction["geometry"]
            for prediction in predictions
        ],
        crs=output_crs,
    )

    prediction_table.to_file(
        cache_path,
        driver="GPKG",
    )

    print(
        f"{arm}/{holdout}/{site}/{stratum}: "
        f"{len(predictions)} candidates saved"
    )

    return predictions

## Fold evaluation

Selects confidence and NMS thresholds on validation AOIs and applies them unchanged to the corresponding held-out site and canopy strata.

In [ ]:
def fold_aois(holdout):
    validation_aois = [
        (site, "mid")
        for site in SITES
        if site != holdout
    ]

    test_aois = [
        (holdout, stratum)
        for stratum in STRATA
    ]

    assert not (
        set(validation_aois)
        & set(test_aois)
    )

    return validation_aois, test_aois


def collect_predictions(
    predictor,
    arm,
    holdout,
    aois,
):
    return [
        prediction
        for site, stratum in aois
        for prediction in predict_aoi(
            predictor,
            arm,
            holdout,
            site,
            stratum,
        )
    ]


def select_thresholds(
    predictions,
    validation_aois,
):
    references = reference_records_for_aois(
        validation_aois
    )
    keep_region = keep_region_for_aois(
        validation_aois
    )

    grid_results = [
        evaluate_at(
            predictions,
            references,
            keep_region,
            confidence,
            nms,
        )
        for confidence in CONFIDENCE_THRESHOLDS
        for nms in NMS_THRESHOLDS
    ]

    # Resolve F1 ties using higher confidence,
    # followed by higher NMS IoU.
    best = max(
        grid_results,
        key=lambda result: (
            result["f1"],
            result["confidence"],
            result["nms"],
        ),
    )

    return best, grid_results


def run_fold(arm, holdout):
    validation_aois, test_aois = fold_aois(
        holdout
    )

    predictor = build_predictor(
        arm,
        holdout,
    )

    validation_predictions = collect_predictions(
        predictor,
        arm,
        holdout,
        validation_aois,
    )

    test_predictions = collect_predictions(
        predictor,
        arm,
        holdout,
        test_aois,
    )

    assert all(
        prediction["site"] != holdout
        for prediction in validation_predictions
    )
    assert all(
        prediction["site"] == holdout
        for prediction in test_predictions
    )

    del predictor
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    selected, validation_grid = select_thresholds(
        validation_predictions,
        validation_aois,
    )

    test_metrics = evaluate_at(
        test_predictions,
        reference_records_for_aois(test_aois),
        keep_region_for_aois(test_aois),
        selected["confidence"],
        selected["nms"],
    )

    stratum_metrics = {}

    for stratum in STRATA:
        stratum_aoi = [(holdout, stratum)]

        stratum_predictions = [
            prediction
            for prediction in test_predictions
            if prediction["stratum"] == stratum
        ]

        stratum_metrics[stratum] = evaluate_at(
            stratum_predictions,
            reference_records_for_aois(
                stratum_aoi
            ),
            keep_region_for_aois(
                stratum_aoi
            ),
            selected["confidence"],
            selected["nms"],
        )

    print(
        f"{ARMS[arm]['label']}/{holdout}: "
        f"confidence={selected['confidence']:.2f}, "
        f"NMS={selected['nms']:.2f}, "
        f"P={test_metrics['precision']:.3f}, "
        f"R={test_metrics['recall']:.3f}, "
        f"F1={test_metrics['f1']:.3f}"
    )

    return {
        "arm": arm,
        "holdout": holdout,
        "selection": selected,
        "validation_grid": validation_grid,
        "test": test_metrics,
        "strata": stratum_metrics,
    }

## Run the H2 evaluation

Evaluates all six fold-specific models through the same multispectral prediction and matching procedure.

In [ ]:
results = {
    "three_band": {},
    "six_band": {},
}

for arm in ("three_band", "six_band"):
    for holdout in SITES:
        print(
            f"\n{ARMS[arm]['label']} | "
            f"hold out {SITE_LABELS[holdout]}"
        )

        results[arm][holdout] = run_fold(
            arm,
            holdout,
        )

## Results and verification

Calculates held-out and macro-averaged performance and verifies the results against the reported H2 values.

In [ ]:
def macro_average(arm_results):
    return {
        metric: float(np.mean([
            arm_results[site]["test"][metric]
            for site in SITES
        ]))
        for metric in (
            "precision",
            "recall",
            "f1",
        )
    }


three_band_macro = macro_average(
    results["three_band"]
)
six_band_macro = macro_average(
    results["six_band"]
)

expected_site_results = {
    "lokoue": {
        "three_band": (0.500, 0.360, 0.418),
        "six_band": (0.474, 0.438, 0.455),
    },
    "dzanga": {
        "three_band": (0.434, 0.471, 0.452),
        "six_band": (0.485, 0.432, 0.457),
    },
    "mbeli": {
        "three_band": (0.347, 0.627, 0.447),
        "six_band": (0.366, 0.516, 0.428),
    },
}

for site in SITES:
    for arm in ("three_band", "six_band"):
        observed = results[
            arm
        ][site]["test"]

        expected = expected_site_results[
            site
        ][arm]

        for metric, expected_value in zip(
            ("precision", "recall", "f1"),
            expected,
        ):
            assert abs(
                observed[metric] - expected_value
            ) < 0.0015, (
                f"{arm}/{site}/{metric}: "
                f"{observed[metric]:.6f} != "
                f"{expected_value:.3f}"
            )

expected_macro = {
    "three_band": (0.427, 0.486, 0.439),
    "six_band": (0.442, 0.462, 0.447),
}

for arm, observed in (
    ("three_band", three_band_macro),
    ("six_band", six_band_macro),
):
    for metric, expected_value in zip(
        ("precision", "recall", "f1"),
        expected_macro[arm],
    ):
        assert abs(
            observed[metric] - expected_value
        ) < 0.0015

expected_stratum_f1 = {
    "lokoue": {
        "tall": (0.426, 0.407),
        "mid": (0.496, 0.519),
        "small": (0.291, 0.425),
    },
    "dzanga": {
        "tall": (0.450, 0.436),
        "mid": (0.433, 0.437),
        "small": (0.475, 0.504),
    },
    "mbeli": {
        "tall": (0.422, 0.437),
        "mid": (0.578, 0.551),
        "small": (0.281, 0.155),
    },
}

for site in SITES:
    for stratum in STRATA:
        expected_three, expected_six = (
            expected_stratum_f1[
                site
            ][stratum]
        )

        observed_three = results[
            "three_band"
        ][site]["strata"][stratum]["f1"]

        observed_six = results[
            "six_band"
        ][site]["strata"][stratum]["f1"]

        assert abs(
            observed_three - expected_three
        ) < 0.0015
        assert abs(
            observed_six - expected_six
        ) < 0.0015

print(
    "PASS: H2 results match the dissertation"
)

## Export H2 results

Exports the site-level, stratum-level, threshold-selection and checkpoint-audit records.

In [ ]:
site_rows = []

for site in SITES:
    three = results["three_band"][site]["test"]
    six = results["six_band"][site]["test"]

    site_rows.append({
        "site": SITE_LABELS[site],
        "three_band_precision": three["precision"],
        "three_band_recall": three["recall"],
        "three_band_f1": three["f1"],
        "six_band_precision": six["precision"],
        "six_band_recall": six["recall"],
        "six_band_f1": six["f1"],
        "delta_f1": (
            round(six["f1"], 3)
            - round(three["f1"], 3)
        ),
    })

site_rows.append({
    "site": "Macro",
    "three_band_precision": (
        three_band_macro["precision"]
    ),
    "three_band_recall": (
        three_band_macro["recall"]
    ),
    "three_band_f1": (
        three_band_macro["f1"]
    ),
    "six_band_precision": (
        six_band_macro["precision"]
    ),
    "six_band_recall": (
        six_band_macro["recall"]
    ),
    "six_band_f1": (
        six_band_macro["f1"]
    ),
    "delta_f1": (
        round(six_band_macro["f1"], 3)
        - round(three_band_macro["f1"], 3)
    ),
})

h2_site_table = pd.DataFrame(site_rows)


stratum_rows = []

for site in SITES:
    for stratum in STRATA:
        three = results[
            "three_band"
        ][site]["strata"][stratum]

        six = results[
            "six_band"
        ][site]["strata"][stratum]

        stratum_rows.append({
            "site": SITE_LABELS[site],
            "stratum": STRATUM_LABELS[stratum],
            "three_band_precision": three["precision"],
            "three_band_recall": three["recall"],
            "three_band_f1": three["f1"],
            "six_band_precision": six["precision"],
            "six_band_recall": six["recall"],
            "six_band_f1": six["f1"],
            "delta_f1": (
                six["f1"] - three["f1"]
            ),
        })

h2_stratum_table = pd.DataFrame(
    stratum_rows
)


threshold_rows = []

for arm in ("three_band", "six_band"):
    for site in SITES:
        selection = results[
            arm
        ][site]["selection"]

        threshold_rows.append({
            "arm": arm,
            "held_out_site": SITE_LABELS[site],
            "confidence": selection["confidence"],
            "nms_iou": selection["nms"],
            "validation_tp": selection["tp"],
            "validation_fp": selection["fp"],
            "validation_fn": selection["fn"],
            "validation_precision": (
                selection["precision"]
            ),
            "validation_recall": (
                selection["recall"]
            ),
            "validation_f1": selection["f1"],
        })

threshold_table = pd.DataFrame(
    threshold_rows
)


h2_site_table.to_csv(
    OUTPUTS / "h2_site_results.csv",
    index=False,
)

h2_stratum_table.to_csv(
    OUTPUTS / "h2_stratum_results.csv",
    index=False,
)

threshold_table.to_csv(
    OUTPUTS / "h2_threshold_selection.csv",
    index=False,
)

model_audit_table.to_csv(
    OUTPUTS / "h2_model_audit.csv",
    index=False,
)

with open(
    OUTPUTS / "results_h2_ms.json",
    "w",
) as output_file:
    json.dump(
        {
            "protocol": {
                "imgmode": "ms",
                "candidate_confidence": (
                    CANDIDATE_THRESHOLD
                ),
                "matching_iou": MATCHING_IOU,
                "confidence_grid": (
                    CONFIDENCE_THRESHOLDS
                ),
                "nms_grid": NMS_THRESHOLDS,
                "threshold_selection": (
                    "per-arm, per-fold validation "
                    "F1; frozen before testing"
                ),
            },
            "three_band": results["three_band"],
            "six_band": results["six_band"],
            "macro": {
                "three_band": three_band_macro,
                "six_band": six_band_macro,
            },
            "model_audit": audit_rows,
        },
        output_file,
        indent=2,
    )

display(h2_site_table.round(3))
display(h2_stratum_table.round(3))
display(threshold_table.round(3))

print("H2 outputs written to:", OUTPUTS)

## Combined results

Combines the verified H1 and H2 outputs to reconstruct Table 4 and the paper’s cross-model figures.

In [ ]:
required_h1_outputs = [
    H1_OUTPUTS / "dataset_composition.csv",
    H1_OUTPUTS / "h1_site_results.csv",
    H1_OUTPUTS / "h1_stratum_results.csv",
]

missing_h1_outputs = [
    path
    for path in required_h1_outputs
    if not path.exists()
]

assert not missing_h1_outputs, (
    "Run H1 evaluation first. Missing:\n"
    + "\n".join(map(str, missing_h1_outputs))
)

composition_table = pd.read_csv(
    H1_OUTPUTS / "dataset_composition.csv"
)
h1_site_table = pd.read_csv(
    H1_OUTPUTS / "h1_site_results.csv"
)
h1_stratum_table = pd.read_csv(
    H1_OUTPUTS / "h1_stratum_results.csv"
)

table_4 = (
    composition_table
    .merge(
        h1_stratum_table[
            [
                "site",
                "stratum",
                "fine_tuned_f1",
            ]
        ],
        on=["site", "stratum"],
        how="inner",
    )
    .merge(
        h2_stratum_table[
            [
                "site",
                "stratum",
                "three_band_f1",
                "six_band_f1",
            ]
        ],
        on=["site", "stratum"],
        how="inner",
    )
    .rename(columns={
        "crowns": "n",
        "density_ha": "density_ha",
        "median_crown_area_m2": (
            "median_area_m2"
        ),
        "fine_tuned_f1": "h1_f1",
        "three_band_f1": "h2_three_band_f1",
        "six_band_f1": "h2_six_band_f1",
    })
)

assert len(table_4) == 9

table_4.to_csv(
    OUTPUTS / "table_4_combined.csv",
    index=False,
)

display(
    table_4[
        [
            "site",
            "stratum",
            "n",
            "density_ha",
            "median_area_m2",
            "h1_f1",
            "h2_three_band_f1",
            "h2_six_band_f1",
        ]
    ].round(3)
)

## Figure 4

Plots macro-averaged precision, recall and F1 for the four evaluated model configurations.

In [ ]:
h1_macro = h1_site_table[
    h1_site_table["site"] == "Macro"
].iloc[0]

h2_macro = h2_site_table[
    h2_site_table["site"] == "Macro"
].iloc[0]

metrics = ["Precision", "Recall", "F1"]

figure_4_values = {
    "Pretrained": [
        h1_macro["pretrained_precision"],
        h1_macro["pretrained_recall"],
        h1_macro["pretrained_f1"],
    ],
    "RGB H1": [
        h1_macro["fine_tuned_precision"],
        h1_macro["fine_tuned_recall"],
        h1_macro["fine_tuned_f1"],
    ],
    "RGB H2": [
        h2_macro["three_band_precision"],
        h2_macro["three_band_recall"],
        h2_macro["three_band_f1"],
    ],
    "Six-band": [
        h2_macro["six_band_precision"],
        h2_macro["six_band_recall"],
        h2_macro["six_band_f1"],
    ],
}

colours = {
    "Pretrained": "#9CA3AF",
    "RGB H1": "#2F6594",
    "RGB H2": "#68A6CC",
    "Six-band": "#DD7627",
}

x = np.arange(len(metrics))
width = 0.18

fig, ax = plt.subplots(
    figsize=(9.2, 4.8)
)

for index, (model, values) in enumerate(
    figure_4_values.items()
):
    positions = (
        x
        + (index - 1.5) * width
    )

    bars = ax.bar(
        positions,
        values,
        width,
        label=model,
        color=colours[model],
    )

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.022,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
        )

ax.set_ylabel(
    "Macro-average score",
    fontweight="bold",
)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1)
ax.grid(
    axis="y",
    color="#D9DEE5",
    linewidth=0.8,
)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.legend(
    ncol=4,
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(0, 1.12),
)

fig.tight_layout()

figure_4_path = (
    OUTPUTS
    / "figure_4_macro_performance.png"
)

fig.savefig(
    figure_4_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Figure 4:", figure_4_path)

## Figure 5

Plots held-out F1 across canopy-height strata for the three fine-tuned model configurations.

In [ ]:
site_order = ["Lokoué", "Dzanga", "Mbeli"]
stratum_order = [
    "High",
    "Intermediate",
    "Low",
]

line_settings = {
    "RGB H1": {
        "column": "h1_f1",
        "colour": "#23649A",
    },
    "RGB H2": {
        "column": "h2_three_band_f1",
        "colour": "#68A9D1",
    },
    "Six-band": {
        "column": "h2_six_band_f1",
        "colour": "#E27424",
    },
}

fig, axes = plt.subplots(
    1,
    3,
    figsize=(10.5, 4.0),
    sharey=True,
)

for axis, site in zip(axes, site_order):
    site_data = (
        table_4[
            table_4["site"] == site
        ]
        .set_index("stratum")
        .loc[stratum_order]
    )

    x = np.arange(len(stratum_order))

    for label, settings in line_settings.items():
        values = site_data[
            settings["column"]
        ].to_numpy()

        axis.plot(
            x,
            values,
            marker="o",
            linewidth=2,
            markersize=7,
            label=label,
            color=settings["colour"],
            markeredgecolor="white",
            markeredgewidth=0.8,
        )

    axis.set_title(
        site,
        loc="left",
        fontweight="bold",
    )
    axis.set_xticks(x)
    axis.set_xticklabels(stratum_order)
    axis.set_ylim(0, 1)
    axis.grid(
        axis="y",
        color="#D9DEE5",
        linewidth=0.8,
    )
    axis.set_axisbelow(True)
    axis.spines[
        ["top", "right", "left", "bottom"]
    ].set_visible(False)
    axis.tick_params(length=0)

axes[0].set_ylabel(
    "F1 score",
    fontweight="bold",
)

axes[2].annotate(
    "0.636",
    xy=(1, 0.636),
    xytext=(1, 0.705),
    ha="center",
    color=colours["RGB H1"],
    fontweight="bold",
)

axes[2].annotate(
    "0.155",
    xy=(2, 0.155),
    xytext=(2, 0.055),
    ha="center",
    color=colours["Six-band"],
    fontweight="bold",
)

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    ncol=3,
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(0.16, 1.05),
)

fig.tight_layout(rect=[0, 0, 1, 0.94])

figure_5_path = (
    OUTPUTS
    / "figure_5_stratum_performance.png"
)

fig.savefig(
    figure_5_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Figure 5:", figure_5_path)